# 算例说明

这一节把后面全程要用的**算例**讲清楚：它是什么、从哪来、有哪些、参数怎么给、结果怎么读。

> 算例（test case）在这里指一个具体的通信测试程序，例如 `reduce_scatter_test`。
> 它负责"发起通信"，HCCL-VM 负责"提供运行环境"，两者配合才完成一次完整的测试。

## 本节学习大纲

- 算例是做什么的
- 算例从哪里来（不需要编译）
- 有哪些算例
- 命令行参数
- 怎么读结果
- 常见问题
- 本节总结

## 1. 算例是做什么的

算例用的是昇腾官方的 HCCL 测试工具 `hccl_test`，它做三件事：

1. **准备数据** —— 按指定的数据量、数据类型生成每个 rank 的输入；
2. **发起通信** —— 调用 HCCL 的集合通信算子（如 ReduceScatter、AllReduce）完成一次通信；
3. **校验结果** —— 把通信结果与预期值比较，给出结论。

在 HCCL-VM 里，它是**发起通信的那一方**：模拟器记录它发出的每一个任务，
Checker 再检查这些任务的依赖、内存访问与同步关系是否正确。

## 2. 算例从哪里来

**算例已经随绿色包提供，不需要编译，也不需要 CANN。**

```
$TEST_BIN/            # 就是 ~/hccl_vm_pack/hccl_test/bin
```

这是本课和"本地版"最大的区别：本地要装 CANN、改 `Makefile`、再 `make` 一遍；
云端直接用包里编好的二进制。

> 为什么可以这样：算例在打包时已经用 aarch64 + CANN 编好，和包里的运行库是配套的。
> 运行时不依赖机器上的 CANN——二进制里写了 RPATH，永远先找包内的 `lib64`。
> 想自己从源码编一次，看 3.9（选看）。

In [ ]:
%%bash
source ~/hccl_vm_pack/env.sh
ls "$TEST_BIN"

应能看到一批以 `_test` 结尾的可执行程序，它们就是全部可用的算例。

## 3. 有哪些算例

命名规律是「算子名 + `_test`」，一个算子对应一个程序：

| 算例 | 对应的集合通信算子 |
| :--- | :--- |
| `all_reduce_test` | AllReduce（全规约）|
| `reduce_scatter_test` | ReduceScatter（规约散射）|
| `all_gather_test` | AllGather（全收集）|
| `alltoall_test` | AlltoAll（全交换）|
| `broadcast_test` | Broadcast（广播）|
| `reduce_test` | Reduce（规约到 root）|
| `scatter_test` / `gather_test` | Scatter / Gather |
| `alltoallv_test` / `reduce_scatterv_test` / `all_gatherv_test` | 变长版本（V 系列）|

**怎么选**：先确定要验证哪个算子，再选同名算例。不确定就先 `ls $TEST_BIN` 看一眼。

本节以 `reduce_scatter_test` 为例，下一节会用到它，便于对照。

## 4. 命令行参数

参数用「短横线 + 单个字母」的形式。以本课要用的命令为例：

```
mpirun --oversubscribe -np 2 $TEST_BIN/reduce_scatter_test \
  -b 64 -e 64 -d int32 -o sum -w 0 -n 1 -c 1
```

`mpirun` 之后、算例之前的参数属于 MPI（`-np 2` 表示启动 2 个进程）；
算例之后的参数才是它自己的。

| 参数 | 含义 | 示例 |
| :--- | :--- | :--- |
| `-b` | 起始数据量（每个 rank 参与通信的字节数）| `-b 64` |
| `-e` | 结束数据量，与 `-b` 配合可做多档扫描 | `-e 64` |
| `-d` | 数据类型 | `int32`、`float16`、`float32` |
| `-o` | 归约操作类型 | `sum`、`max`、`min`、`prod` |
| `-w` | 预热迭代次数，不计入统计 | `-w 0` |
| `-n` | 正式迭代次数 | `-n 1` |
| `-c` | 是否做数值校验：`1` 校验、`0` 不校验 | `-c 1` |
| `-f` | 数据量增长倍数（多档扫描时使用）| — |
| `-p` | 每个 rank 上的进程数 | — |
| `-r` | 参与通信的 rank 数量 | — |

几点说明：

- **`-b` 与 `-e` 相同**时只测这一个数据量；不同时按 `-f` 的倍数从 `-b` 扫到 `-e`；
- **`-w 0`** 表示不预热。数据量小通常不需要预热；数据量大、想看稳定性能时给几次；
- **`-c 1`** 会额外跑一次通信来比对结果，所以日志里会看到**两条算子记录**；
- **`-o`** 只在带规约语义的算子上有效（AllReduce、Reduce、ReduceScatter），
  像 AllGather 这种不涉及归约的算子，这个参数没有意义。

## 5. 怎么读结果

算例会打印一张表：

| 字段 | 含义 |
| :--- | :--- |
| `data_size(Bytes)` | 本次测试的数据量 |
| `aveg_time(us)` | 平均耗时（微秒）|
| `alg_bandwidth(GB/s)` | 算法带宽（GB/s）|
| `check_result` | 数值校验结论 |

**在模拟环境里，`check_result` 显示 `failed` 是正常的**，原因有两个：

- HCCL-VM 是**任务图模拟器**，记录并检查"谁给谁发了什么、依赖对不对"，但**不真做数值计算**，
  所以缓冲区里的数据不会被真正加起来；
- `aveg_time` 恒为 `1000.00 us` 也是占位值，不是真实耗时。

**那怎么判断成功？看 Checker。** 下一节会用 `plugin run @checker` 检查任务图，
输出 `All Success` 才算通过。数值校验在本课里只作为"算子确实被调用过"的旁证。

> 模拟环境的耗时与带宽不能代表真实 NPU 的性能，只能用于观察趋势和对比不同配置。

## 6. 常见问题

**`ls $TEST_BIN` 提示没有这个目录**
说明绿色包没解压，或者 `env.sh` 没生成。回到 03.03 按顺序跑一遍。

**运行时报 rank 数量与通信域不匹配**
`mpirun -np` 的进程数要和通信域里的 rank 数一致。通信域 `112` 是 2 个 rank，`-np` 就写 `2`。

**`check_result` 是 `failed`**
在模拟环境里属于正常，判断成败看 Checker（下一节）。

**日志里出现大量告警**
模拟环境没有真实硬件，很多分支会打印告警后继续执行。
判断顺序是：**先看 Checker 的汇总结论，再看告警**——告警用于定位，不用于判定成败。

**想自己编一份算例**
云端包里没有源码。需要改算例或重新编译时，看 3.9 的本地环境。

## 7. 本节总结

- 算例负责**准备数据、发起通信**，模拟器负责提供环境、记录任务，Checker 负责校验任务图；
- 云端版的算例**随绿色包提供**，不需要装 CANN、不需要编译；
- 按算子选算例（`<算子名>_test`），最需要留意 `-b` 数据量、`-d` 类型、`-o` 归约方式、`-c` 是否校验；
- 模拟环境里 `check_result` 显示 `failed` 是正常的，**成败看 Checker 的 `All Success`**。

[← 3.3 获取并解压绿色包](./03.03_cloud_pack_setup.ipynb) · [返回本章目录](./03.01_chapter_intro.ipynb) · [3.5 跑通第一个通信测试 →](./03.05_first_comm_test.ipynb)